# Length-Scale Selection for Job Corps Data (Semi-Synthetic)

This notebook reproduces the length-scale and ridge parameter tuning experiments for the Job Corps dataset.

**Workflow:**
1. **Data Loading:** Loads the Job Corps empirical application data (`emp_app.csv`) and performs one-hot encoding.
2. **Range Estimation:** Uses median pairwise distances to derive heuristic ranges for length-scales
   (\(\ell_X\) for covariates and \(\ell_T\) for treatment).
3. **Hyperparameter Tuning:** Runs a 2D grid search with Nyström KRR and LOOCV to select \(\ell_X\), \(\ell_T\),
   and the ridge parameter \(eta\).


In [6]:
import sys
import pathlib
import numpy as np
import pandas as pd
import logging
from typing import Tuple, List, Dict, Optional, Union

# --- Environment Setup ---
# If running in Colab, mount Drive; otherwise assume local paths.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = pathlib.Path("/content/drive/MyDrive/Colab Notebooks/CTE_Codes")
except ImportError:
    BASE_DIR = pathlib.Path(".").resolve()

sys.path.append(str(BASE_DIR))

# Project-specific imports
from KRR_methods.data_jobcorps import make_Xss
from KRR_methods.algorithms.length_selection import tune_length2d_and_beta_loocv_krr_nystrom

print(f"Working Directory: {BASE_DIR}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Working Directory: /content/drive/MyDrive/Colab Notebooks/CTE_Codes


In [7]:
def load_and_preprocess_jobcorps(base_dir: pathlib.Path) -> Tuple[pd.DataFrame, pd.Series, pd.Series]:
    """Load Job Corps data and perform one-hot encoding.

    Returns:
        X (pd.DataFrame): Covariates after preprocessing.
        T (pd.Series): Treatment variable (column "d").
        Y (pd.Series): Outcome variable (column "y").
    """
    emp_dir = base_dir / "DML_methods" / "Data_and_Results"
    data_path = emp_dir / "emp_app.csv"

    if not data_path.exists():
        raise FileNotFoundError(f"Data file not found at: {data_path}")

    print(f"Loading data from: {data_path.name}...")
    data = pd.read_csv(data_path, index_col=0)

    # Shuffle once (fixed seed) to match the original script
    data = data.sample(frac=1, random_state=20)

    # One-hot encoding for categorical (int64) columns
    data_processed = pd.concat(
        [
            data.select_dtypes(exclude="int64"),
            pd.get_dummies(
                data.select_dtypes("int64").astype("category"),
                drop_first=True,
                dtype=float,
            ),
        ],
        axis=1,
    )

    X = data_processed.drop(["d", "y"], axis=1)  # Covariates
    T = data_processed["d"]                      # Treatment
    Y = data_processed["y"]                      # Outcome

    print(f"Data Loaded. Shapes -> X: {X.shape}, T: {T.shape}, Y: {Y.shape}")
    return X, T, Y

# Execute loading
X, T, Y = load_and_preprocess_jobcorps(BASE_DIR)

# Standardize covariates (min-max scaling) for kernel distance calculations
Xss = make_Xss(X)


Loading data from: emp_app.csv...
Data Loaded. Shapes -> X: (4024, 138), T: (4024,), Y: (4024,)


In [8]:
# ============================================================
# Utility Functions for Length-Scale Heuristics
# ============================================================

def pairwise_dist_median(Z: np.ndarray, max_pairs: int = 200_000, seed: int = 123) -> float:
    """Median of pairwise Euclidean distances (with subsampling if needed).

    Args:
        Z: Input array of shape (n, d)
        max_pairs: Threshold for exact vs. subsampled computation.
        seed: RNG seed for subsampling.
    """
    n = Z.shape[0]
    rng = np.random.default_rng(seed)
    total_pairs = n * (n - 1) // 2

    if total_pairs <= max_pairs:
        G = Z @ Z.T
        sq = np.sum(Z * Z, axis=1, keepdims=True)
        D2 = np.maximum(sq + sq.T - 2.0 * G, 0.0)
        iu = np.triu_indices(n, k=1)
        dists = np.sqrt(D2[iu], dtype=Z.dtype)
        return float(np.median(dists))
    else:
        m = max_pairs
        i = rng.integers(0, n, size=m)
        j = rng.integers(0, n, size=m)
        same = (i == j)
        if np.any(same):
            j[same] = (j[same] + 1) % n
        dists = np.linalg.norm(Z[i] - Z[j], axis=1)
        return float(np.median(dists))


def matern_c(nu: float) -> float:
    """Scaling constant c(nu) for Matérn kernels where z = c * r / ell."""
    if nu <= 0:
        raise ValueError("nu must be positive.")
    return float(np.sqrt(2.0 * nu))


def solve_z_for_tau(rho: float, nu: float = 1.5, tol: float = 1e-12, max_iter: int = 200) -> float:
    """Solve for z > 0 such that tau(z; nu) = rho for Matérn kernels."""
    import math

    if not (0.0 < rho < 1.0):
        raise ValueError(f"rho must be in (0, 1), got {rho}")

    if abs(nu - 0.5) < 1e-9:
        return -math.log(rho)

    if abs(nu - 1.5) < 1e-9:
        def P(z): return 1.0 + z
    elif abs(nu - 2.5) < 1e-9:
        def P(z): return 1.0 + z + (z**2) / 3.0
    else:
        raise ValueError("solve_z_for_tau currently supports nu in {0.5, 1.5, 2.5}.")

    def f(z):
        return P(z) * math.exp(-z) - rho

    z_lo, z_hi = 0.0, 50.0
    if f(z_lo) < 0:
        return z_lo

    while f(z_hi) > 0 and z_hi < 1e6:
        z_hi *= 2.0

    for _ in range(max_iter):
        z_mid = 0.5 * (z_lo + z_hi)
        if f(z_mid) > 0:
            z_lo = z_mid
        else:
            z_hi = z_mid
        if (z_hi - z_lo) < tol:
            break

    return 0.5 * (z_lo + z_hi)


In [9]:
# ============================================================
# Wrappers for Estimating Length Scales (ell) for X and T
# ============================================================

def estimate_ell_for_jobcorps_X_only(
    X_origin, rhos=(0.5, 0.8), nu=1.5, kernel_type="matern",
    max_pairs=200_000, seed=42, x_divisor=1
) -> dict:
    """Estimate ell values for X using median distance + target correlations."""
    X_orig_np = np.asarray(X_origin, dtype=float)
    X_scaled = X_orig_np / float(x_divisor)

    r_med = pairwise_dist_median(X_scaled, max_pairs=max_pairs, seed=seed)
    kernel_type = str(kernel_type).lower()
    results = []

    if kernel_type == "matern":
        c = matern_c(nu)
        for rho in rhos:
            z_rho = solve_z_for_tau(rho, nu=nu)
            ell_rho = (c * r_med) / z_rho
            results.append({
                "rho": float(rho), "z_rho": float(z_rho),
                "ell_rho": float(ell_rho), "kernel_type": "matern", "nu": float(nu)
            })
        out_nu = float(nu)

    elif kernel_type == "gaussian":
        for rho in rhos:
            z_rho = np.sqrt(-np.log(rho))
            ell_rho = r_med / z_rho
            results.append({
                "rho": float(rho), "z_rho": float(z_rho),
                "ell_rho": float(ell_rho), "kernel_type": "gaussian"
            })
        out_nu = None
    else:
        raise ValueError("kernel_type must be 'matern' or 'gaussian'.")

    return {"r_median": float(r_med), "kernel_type": kernel_type, "nu": out_nu, "results": results}


def estimate_ell_for_T_only(
    T_origin, rhos=(0.5, 0.8), nu=1.5, kernel_type="matern",
    max_pairs=200_000, seed=42, t_divisor=1
) -> dict:
    """Estimate ell values for T using median distance + target correlations."""
    T_arr = np.asarray(T_origin, dtype=float).reshape(-1, 1)
    T_scaled = T_arr / float(t_divisor)
    r_med = pairwise_dist_median(T_scaled, max_pairs=max_pairs, seed=seed)

    kernel_type = str(kernel_type).lower()
    results = []

    if kernel_type == "laplace":
        # exp(-r/ell) = rho
        for rho in rhos:
            z_rho = solve_z_for_tau(rho, nu=0.5)
            ell_rho = r_med / z_rho
            results.append({
                "rho": float(rho), "z_rho": float(z_rho),
                "ell_rho": float(ell_rho), "kernel_type": "laplace", "nu": None
            })
        out_nu = None

    elif kernel_type == "matern":
        c = matern_c(nu)
        for rho in rhos:
            z_rho = solve_z_for_tau(rho, nu=nu)
            ell_rho = (c * r_med) / z_rho
            results.append({
                "rho": float(rho), "z_rho": float(z_rho),
                "ell_rho": float(ell_rho), "kernel_type": "matern", "nu": float(nu)
            })
        out_nu = float(nu)

    elif kernel_type == "gaussian":
        for rho in rhos:
            z_rho = np.sqrt(-np.log(rho))
            ell_rho = r_med / z_rho
            results.append({
                "rho": float(rho), "z_rho": float(z_rho),
                "ell_rho": float(ell_rho), "kernel_type": "gaussian", "nu": None
            })
        out_nu = None
    else:
        raise ValueError("kernel_type must be one of {'laplace', 'matern', 'gaussian'}.")

    return {"r_median": float(r_med), "kernel_type": kernel_type, "nu": out_nu, "results": results}


## Step 1: Median-based Range Estimation

Before running the grid search, estimate plausible ranges for the length-scales by
matching median pairwise distances to target correlations (\(
ho\)).


In [10]:
# 1. Estimate ranges for X (covariates)
print("--- Estimating Range for X (Matern, nu=0.5) ---")
out_x = estimate_ell_for_jobcorps_X_only(Xss, rhos=(0.2, 0.85), nu=0.5, kernel_type="matern")
print(out_x)

# 2. Estimate ranges for T (treatment)
print("\n--- Estimating Range for T (Laplace) ---")
out_t = estimate_ell_for_T_only(T, rhos=(0.15, 0.85), kernel_type="laplace")
print("Laplace:", out_t)


--- Estimating Range for X (Matern, nu=0.5) ---
{'r_median': 4.69041575982343, 'kernel_type': 'matern', 'nu': 0.5, 'results': [{'rho': 0.2, 'z_rho': 1.6094379124341003, 'ell_rho': 2.9143191691872627, 'kernel_type': 'matern', 'nu': 0.5}, {'rho': 0.85, 'z_rho': 0.16251892949777494, 'ell_rho': 28.86073501910217, 'kernel_type': 'matern', 'nu': 0.5}]}

--- Estimating Range for T (Laplace) ---
Laplace: {'r_median': 840.71428535, 'kernel_type': 'laplace', 'nu': None, 'results': [{'rho': 0.15, 'z_rho': 1.8971199848858813, 'ell_rho': 443.1529328918919, 'kernel_type': 'laplace', 'nu': None}, {'rho': 0.85, 'z_rho': 0.16251892949777494, 'ell_rho': 5173.023769895742, 'kernel_type': 'laplace', 'nu': None}]}


## Step 2: 2D Grid Search with LOOCV

We run a grid search over \(\ell_X\) and \(\ell_T\). For each pair,
we optimize the ridge parameter \(eta\) using LOOCV with a Nyström approximation.

- **Kernel:** Product kernel \(k((x,t),(x',t')) = k_X(x,x') \cdot k_T(t,t')\)
- **Approximation:** Nyström method with `m=700` landmarks


In [11]:
def main() -> None:
    # Use the semi-synthetic X generated earlier
    Xss_data = make_Xss(X)

    print("\n### Laplace kernel (Matern nu=0.5) — Grid Search Execution")

    # Grid definition based on the heuristic results from Step 1
    length_grid_x = [7, 9, 11, 13, 15, 17, 19, 21, 23, 25, 27, 29]  # ell_X
    length_grid_t = [250, 500, 1000, 2000, 3000, 4000, 5000, 6000]  # ell_T

    # Run the 2D tuning (beta is optimized internally for each pair)
    res_2d = tune_length2d_and_beta_loocv_krr_nystrom(
        Xss_data,
        T,
        Y,
        length_grid_x=length_grid_x,
        length_grid_t=length_grid_t,
        m=700,                      # Nyström landmarks
        kernel_type_f="matern",
        nu_x=0.5,                   # Roughness for X (0.5 = Laplace-like)
        nu_t=0.5,                   # Roughness for T
        beta_bounds=(1e-4, 1e2),
        random_state=0,
    )

    print("\n" + "="*40)
    print("Best 2D Hyperparameter Result:")
    print("="*40)
    print(f"  ell_x* = {res_2d['best_length_x']}")
    print(f"  ell_t* = {res_2d['best_length_t']}")
    print(f"  beta* = {res_2d['best_beta']:.6f}")
    print(f"  lambda* = {res_2d['best_lambda']:.6e}")
    print(f"  MSE* = {res_2d['best_mse']:.4f}")

if __name__ == "__main__":
    main()



### Laplace kernel (Matern nu=0.5) — Grid Search Execution
Best ell_x = 13.0, ell_t = 6000.0, beta* = 0.3178656453779402, lambda* = 7.899245660485591e-05, LOOCV MSE = 1305.6361993509345

Best 2D Hyperparameter Result:
  ell_x* = 13.0
  ell_t* = 6000.0
  beta* = 0.317866
  lambda* = 7.899246e-05
  MSE* = 1305.6362


# T-to-Y Regression: Length Parameter Selection

This section tunes a **T-only** KRR baseline by reusing the 2D tuner with a
constant dummy X so the product kernel reduces to the T-kernel.


In [12]:
import numpy as np
import io
import contextlib

from KRR_methods.algorithms.length_selection import tune_length2d_and_beta_loocv_krr_nystrom


def tune_t_only_length_loocv_nystrom(
    T,
    Y,
    length_grid_t,
    *,
    m: int = 700,
    kernel_type_f: str = "matern",
    nu_t: float = 0.5,
    beta_bounds=(1e-4, 1e2),
    random_state: int = 0,
    suppress_internal_prints: bool = True,
):
    """T-only LOOCV length selection using Nyström approximation.

    Key trick:
      - Use a constant dummy X so k_X == 1 for stationary kernels.
      - The product kernel then reduces to the T-kernel.
    """
    T_arr = np.asarray(T, dtype=float).reshape(-1)
    Y_arr = np.asarray(Y, dtype=float).reshape(-1)
    n = T_arr.shape[0]

    # Constant dummy X so k_X(x_i, x_j) == 1
    X_dummy = np.zeros((n, 1), dtype=float)

    # ell_x is meaningless here; keep it singleton to minimize compute
    length_grid_x_dummy = [1.0]

    # Nyström landmarks cannot exceed n
    m_eff = int(min(m, n))

    if suppress_internal_prints:
        buf_out = io.StringIO()
        buf_err = io.StringIO()
        with contextlib.redirect_stdout(buf_out), contextlib.redirect_stderr(buf_err):
            res_raw = tune_length2d_and_beta_loocv_krr_nystrom(
                X_dummy,
                T_arr,
                Y_arr,
                length_grid_x=length_grid_x_dummy,
                length_grid_t=list(length_grid_t),
                m=m_eff,
                kernel_type_f=kernel_type_f,
                nu_x=0.5,               # arbitrary; X is dummy
                nu_t=float(nu_t),
                beta_bounds=beta_bounds,
                random_state=random_state,
            )
    else:
        res_raw = tune_length2d_and_beta_loocv_krr_nystrom(
            X_dummy,
            T_arr,
            Y_arr,
            length_grid_x=length_grid_x_dummy,
            length_grid_t=list(length_grid_t),
            m=m_eff,
            kernel_type_f=kernel_type_f,
            nu_x=0.5,
            nu_t=float(nu_t),
            beta_bounds=beta_bounds,
            random_state=random_state,
        )

    res_clean = {
        "best_length_t": res_raw["best_length_t"],
        "best_beta": res_raw["best_beta"],
        "best_mse": res_raw["best_mse"],
        "nystrom_m": m_eff,
        "nu_t": float(nu_t),
        "beta_bounds": beta_bounds,
        "full_result": res_raw,
    }
    return res_clean


# -------------------------------------------------------------------
# Usage (after T and Y are defined)
# -------------------------------------------------------------------
length_grid_t = [250, 500, 1000, 2000, 3000, 4000, 5000, 6000]

res_t_only = tune_t_only_length_loocv_nystrom(
    T=T,
    Y=Y,
    length_grid_t=length_grid_t,
    m=700,
    kernel_type_f="matern",
    nu_t=0.5,                 # change to 1.5 for Matérn(ν=1.5) on T
    beta_bounds=(1e-4, 1e2),
    random_state=0,
    suppress_internal_prints=True,
)

print("\n" + "=" * 40)
print("Best T-only Nyström-LOOCV Result")
print("=" * 40)
print(f"  ell_t* = {res_t_only['best_length_t']}")
print(f"  beta*  = {res_t_only['best_beta']:.6f}")
print(f"  MSE*   = {res_t_only['best_mse']:.6e}")
print(f"  m      = {res_t_only['nystrom_m']}")



Best T-only Nyström-LOOCV Result
  ell_t* = 3000.0
  beta*  = 3.928282
  MSE*   = 1.413240e+03
  m      = 700
